In [ ]:
#!/usr/bin/env python3
"""
download_libsvm_binary_parallel.py

Download all dataset files linked from:
  https://www.csie.ntu.edu.tw/~cjlin/libsvmtools/datasets/binary.html

What it does
- Fetches the HTML page.
- Extracts all <a href="..."> links that point to files under the corresponding
  /datasets/binary/ directory on csie.ntu.edu.tw (scheme can be http or https).
- Downloads each file in parallel using a thread pool.
- Best-effort resume support using HTTP Range requests; partial downloads are stored
  as *.part and moved into place when complete.
- Retries failed downloads with exponential backoff.

Notes
- Some datasets on that page are extremely large. Ensure you have sufficient disk space
  and bandwidth before running.
- This script uses only the Python standard library.

Example usage
  python3 download_libsvm_binary_parallel.py --out libsvm_binary --workers 16

Optional: include external links (not on csie.ntu.edu.tw /datasets/binary/)
  python3 download_libsvm_binary_parallel.py --include-external
"""

from __future__ import annotations

import argparse
import os
import sys

# Drop IPython/Jupyter argv when run inside a notebook so argparse does not error.
if __name__ == "__main__" and "ipykernel" in sys.modules:
    sys.argv = sys.argv[:1]
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from html.parser import HTMLParser
from typing import List, Set, Tuple
from urllib.error import HTTPError, URLError
from urllib.parse import urljoin, urlparse, unquote
from urllib.request import Request, urlopen


DEFAULT_PAGE = "https://www.csie.ntu.edu.tw/~cjlin/libsvmtools/datasets/binary.html"
USER_AGENT = "Mozilla/5.0 (compatible; libsvm-binary-downloader/1.0)"


class _LinkParser(HTMLParser):
    def __init__(self) -> None:
        super().__init__()
        self.hrefs: Set[str] = set()

    def handle_starttag(self, tag: str, attrs: List[Tuple[str, str]]) -> None:
        if tag.lower() != "a":
            return
        for k, v in attrs:
            if k.lower() == "href" and v:
                self.hrefs.add(v.strip())


def _read_url_text(url: str, timeout: float) -> str:
    req = Request(url, headers={"User-Agent": USER_AGENT})
    with urlopen(req, timeout=timeout) as resp:
        data = resp.read()
    return data.decode("utf-8", "ignore")


def _binary_dir_parts(page_url: str) -> Tuple[str, str]:
    """
    Return (netloc, path_prefix) for the /datasets/binary/ directory that corresponds
    to the given binary.html page URL.
    """
    binary_dir_url = urljoin(page_url, "binary/")
    p = urlparse(binary_dir_url)
    return p.netloc, p.path  # e.g. "www.csie.ntu.edu.tw", "/~cjlin/libsvmtools/datasets/binary/"


def _is_downloadable_link(href: str) -> bool:
    if not href:
        return False
    if href.startswith("#"):
        return False
    if href.lower().startswith("mailto:"):
        return False
    return True


def extract_file_urls(
    page_url: str,
    timeout: float,
    include_external: bool = False,
) -> List[str]:
    """
    Extract file URLs from the page.

    By default, we only keep URLs that:
      - are hosted on the same netloc as the /datasets/binary/ directory, and
      - whose path starts with that directory path, and
      - do not end with "/" or ".html"

    If include_external=True, we keep any http(s) URL that does not end with "/" or ".html".
    """
    html = _read_url_text(page_url, timeout=timeout)

    parser = _LinkParser()
    parser.feed(html)

    keep_netloc, keep_path_prefix = _binary_dir_parts(page_url)
    urls: Set[str] = set()

    for href in parser.hrefs:
        if not _is_downloadable_link(href):
            continue

        u = urljoin(page_url, href)
        pu = urlparse(u)

        if pu.scheme not in ("http", "https"):
            continue
        if pu.path.endswith("/") or pu.path.endswith(".html"):
            continue

        if not include_external:
            if pu.netloc != keep_netloc:
                continue
            if not pu.path.startswith(keep_path_prefix):
                continue

        urls.add(u)

    return sorted(urls)


def _filename_from_url(url: str) -> str:
    pu = urlparse(url)
    name = os.path.basename(pu.path)
    name = unquote(name)
    # Extra safety: avoid empty names and path traversal
    name = name.replace(os.sep, "_").replace("/", "_").strip()
    return name


@dataclass(frozen=True)
class DownloadResult:
    url: str
    dest_path: str
    ok: bool
    message: str


def download_one(
    url: str,
    out_dir: str,
    *,
    chunk_size: int,
    timeout: float,
    retries: int,
    resume: bool,
    force: bool,
) -> DownloadResult:
    """
    Download a single URL to out_dir in a streaming manner.

    Resume behavior:
    - If dest exists and not force: skip.
    - If dest exists and force: overwrite.
    - Partial downloads are written to dest + ".part".
    - If .part exists and resume: attempt Range download from its size.
    """
    filename = _filename_from_url(url)
    if not filename:
        return DownloadResult(url, "", False, "Could not infer filename from URL")

    os.makedirs(out_dir, exist_ok=True)
    dest = os.path.join(out_dir, filename)
    tmp = dest + ".part"

    if os.path.exists(dest) and not force:
        return DownloadResult(url, dest, True, "already exists (skipped)")

    if force:
        # Remove any previous files to avoid confusion.
        try:
            if os.path.exists(dest):
                os.remove(dest)
        except OSError:
            pass

    attempt = 0
    while True:
        attempt += 1

        # Determine resume offset (if any)
        offset = 0
        if resume and os.path.exists(tmp):
            try:
                offset = os.path.getsize(tmp)
            except OSError:
                offset = 0

        headers = {
            "User-Agent": USER_AGENT,
            "Accept-Encoding": "identity",
        }
        if resume and offset > 0:
            headers["Range"] = f"bytes={offset}-"

        try:
            req = Request(url, headers=headers)

            # urlopen raises HTTPError for 4xx/5xx; otherwise returns a response.
            with urlopen(req, timeout=timeout) as resp:
                code = getattr(resp, "status", None) or resp.getcode()

                # If we asked for Range but server ignored it (200), restart from scratch.
                if offset > 0 and code == 200:
                    offset = 0

                mode = "ab" if (offset > 0 and code == 206) else "wb"

                # Stream to tmp
                with open(tmp, mode) as f:
                    while True:
                        chunk = resp.read(chunk_size)
                        if not chunk:
                            break
                        f.write(chunk)

            # Move into place atomically
            os.replace(tmp, dest)
            return DownloadResult(url, dest, True, "downloaded")

        except HTTPError as e:
            # Special-case: Range Not Satisfiable; restart from scratch
            if e.code == 416 and resume:
                try:
                    if os.path.exists(tmp):
                        os.remove(tmp)
                except OSError:
                    pass
                # Retry immediately (counts as an attempt)
                if attempt <= retries:
                    continue

            if attempt > retries:
                return DownloadResult(url, dest, False, f"HTTPError {e.code}: {e.reason}")
        except (URLError, TimeoutError, OSError) as e:
            if attempt > retries:
                return DownloadResult(url, dest, False, f"{type(e).__name__}: {e}")

        # Backoff before next attempt
        if attempt <= retries:
            sleep_s = min(60.0, 2.0 ** (attempt - 1))
            time.sleep(sleep_s)
        else:
            return DownloadResult(url, dest, False, "Failed after retries")

In [ ]:
def main() -> int:
    ap = argparse.ArgumentParser(
        description="Parallel downloader for LIBSVM binary datasets page."
    )
    ap.add_argument("--page", default=DEFAULT_PAGE, help="Dataset page URL (binary.html).")
    ap.add_argument("--out", default="libsvm_binary", help="Output directory.")
    ap.add_argument(
        "--workers",
        type=int,
        default=min(32, (os.cpu_count() or 4) * 4),
        help="Number of parallel download workers (threads).",
    )
    ap.add_argument("--timeout", type=float, default=60.0, help="Network timeout (seconds).")
    ap.add_argument("--retries", type=int, default=5, help="Retries per file.")
    ap.add_argument(
        "--chunk-size",
        type=int,
        default=1024 * 1024,
        help="Read chunk size in bytes (default: 1 MiB).",
    )
    ap.add_argument(
        "--no-resume",
        action="store_true",
        help="Disable resume; always restart downloads.",
    )
    ap.add_argument(
        "--force",
        action="store_true",
        help="Overwrite existing completed files in output directory.",
    )
    ap.add_argument(
        "--include-external",
        action="store_true",
        help="Also download external links (not hosted under csie.ntu.edu.tw /datasets/binary/).",
    )
    ap.add_argument(
        "--list",
        action="store_true",
        help="List discovered URLs and exit (no downloading).",
    )
    args = ap.parse_args()

    try:
        urls = extract_file_urls(
            args.page,
            timeout=args.timeout,
            include_external=args.include_external,
        )
    except Exception as e:
        print(f"Failed to parse page: {e}", file=sys.stderr)
        return 2

    if not urls:
        print("No downloadable URLs found.", file=sys.stderr)
        return 1

    if args.list:
        for u in urls:
            print(u)
        return 0

    print(f"Found {len(urls)} file(s). Output dir: {args.out}")
    print(f"Workers: {args.workers} | Resume: {not args.no_resume} | Retries: {args.retries}")

    start = time.time()
    ok_count = 0
    fail_count = 0

    # Submit downloads
    with ThreadPoolExecutor(max_workers=max(1, args.workers)) as ex:
        futures = [
            ex.submit(
                download_one,
                u,
                args.out,
                chunk_size=args.chunk_size,
                timeout=args.timeout,
                retries=args.retries,
                resume=not args.no_resume,
                force=args.force,
            )
            for u in urls
        ]

        for i, fut in enumerate(as_completed(futures), start=1):
            res = fut.result()
            status = "OK" if res.ok else "FAIL"
            print(f"[{i}/{len(urls)}] {status} {res.url} -> {res.message}")

            if res.ok:
                ok_count += 1
            else:
                fail_count += 1

    elapsed = time.time() - start
    print(f"Done. ok={ok_count} fail={fail_count} elapsed={elapsed:.1f}s")

    return 0 if fail_count == 0 else 3


if __name__ == "__main__":
    raise SystemExit(main())
